# Phase 1 — Parse Documents and Preserve Evidence

**Roadmap reference:** `ContractIQ_Project_Execution_Roadmap.md`, Phase 1

**Goal:** parse real DOCX contracts into paragraph-level clause records and table records, each with a stable, re-locatable source location (`paragraph:14`, `table:2,row:4,cell:3`).

**Guiding rule:** start with ONE representative MSA and ONE table-heavy supporting document. Only run the full 36-document batch after manually reviewing these.

**Completion gate:** every extracted clause can be located in the original document; tables preserve real cell values; failures are recorded per file, not swallowed; no source document is modified.

## 1. Load configuration and the Phase 0 manifest

In [ ]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq.ingestion import load_manifest
from src.contractiq.parsing import (
    run_phase1_sample,
    run_phase1_full,
    load_parsed_document,
    table_cell_location,
)
from src.contractiq.parsing import run_phase1_sample,run_

manifest = load_manifest()
print(f"Loaded manifest: {len(manifest)} documents")

if len(manifest) != config.EXPECTED_TOTAL_DOCUMENTS:
    print(f"⚠ Expected {config.EXPECTED_TOTAL_DOCUMENTS}, got {len(manifest)}. "
          "Re-run 00_Project_Setup.ipynb before continuing.")

ImportError: cannot import name 'run_phase1_sample' from 'src.contractiq.parsing' (e:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\Research\..\src\contractiq\parsing.py)

## 2. Pick one representative MSA and one table-heavy supporting document

Look at the printed manifest below and choose two real `doc_id` values — one from `master_agreements` (an MSA) and one that's likely to contain tables (e.g. a rate card in `commercial_docs`, or an invoice in `financial_billing`). Update `SAMPLE_DOC_IDS` in the next cell with the actual IDs you pick.

In [ ]:
for r in manifest:
    if r.category_key in ("master_agreements", "commercial_docs", "financial_billing"):
        print(f"  {r.doc_id:<10} {r.category_key:<20} {r.filename}")

## 3. Parse the sample — manual verification step

This does NOT touch the other 34 documents yet. Review the printed clause/table counts and the location-verification result before moving on.

In [ ]:
# TODO: replace with the two doc_ids you picked above
SAMPLE_DOC_IDS = ["MLA-001", "COM-001"]

sample_results = run_phase1_sample(manifest, SAMPLE_DOC_IDS)

## 4. Manually inspect a few extracted clauses

Read these against the actual Word document open side-by-side. This is the human-verification step the roadmap requires before scaling up — not optional, and not replaceable by the automated check above alone.

In [ ]:
sample_id = SAMPLE_DOC_IDS[0]
parsed = load_parsed_document(sample_id)

print(f"First 5 clauses of {sample_id}:\n")
for clause in parsed["clauses"][:5]:
    print(f"[{clause['source_location']['locator']}] "
          f"heading={clause['heading']!r} hint={clause['heading_hint']!r}")
    print(f"    {clause['text'][:120]}")
    print()

In [ ]:
print(f"Tables found in {sample_id}: {len(parsed['tables'])}\n")
if parsed["tables"]:
    first_table = parsed["tables"][0]
    print(f"Table 0 has {len(first_table['rows'])} rows:")
    for row in first_table["rows"][:5]:
        print("   ", row)

## 5. Only after the sample looks correct: parse the full manifest

Uncomment and run once you've manually confirmed step 4 is accurate. This processes all 36 documents; failures are reported per file and don't stop the batch.

In [ ]:
# full_results = run_phase1_full(manifest)

---
### Next: `02_Entity_Clause_and_EDA.ipynb`

Extract parties, dates, monetary amounts, and contract-specific terms from these clause records, keeping raw text and normalized values separate and source-linked.